<a href="https://colab.research.google.com/github/Davidson-Open-University/phy107-labs/blob/main/notebooks/PHY107-Lab9-Lubrication-Viscosity-and-Bearings.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PHY 107 Lab 9: Lubrication, Viscosity and Bearings

**Unit 9: Lubrication and bearings** · General Practical Physics I · Davidson Open University

**In this lab you will:**
- compare dry and lubricated friction from data;
- describe how viscosity changes with temperature from a graph;
- calculate power lost to friction in a bearing;
- fit an exponential model to viscosity data.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the online lab and the examples, and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short analysis function and test it on data.
- **Part D – Your own data and reflection** (about 10 minutes): analyse readings you took in the online lab (or at home), and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `phy107-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

**Honest data:** keep your own readings separate from the example data, and label every simulated or supplied reading as such. Never invent readings.

*This lab is practice to help you learn and prepare for the unit assignment (lab report). It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False, dp=6):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), dp) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i, False, dp) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i, False, dp) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k, False, dp) + '=' + _norm(v, False, dp) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), dp) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('phy107|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False, spec.get('dp', 6))
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False), spec.get('dp', 6))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '9.1': {'h': 'a0e66b476466bfab', 'hint': 'μ = mean pulling force / normal force, for each set.', 'dp': 2},
    '9.2': {'h': 'a4b159d4639f53bf', 'hint': '(dry − oiled) / dry × 100, using the mean forces.', 'ok': 'Lubrication cut friction by more than half.', 'dp': 1},
    '9.3': {'h': '3d93feef7995e6df', 'hint': '(η at 20 °C − η at 80 °C) / η at 20 °C × 100.', 'ok': 'This is why engines need the right grade of oil for the climate.', 'dp': 1},
    '9.4': {'h': '30c230c1d4d2f6ab', 'hint': 'P = F * v.', 'ok': 'That energy becomes heat in the bearing.', 'dp': 1},
    '9.5': {'h': '51266ba7acd732fd', 'hint': 'b = −np.polyfit(temp, np.log(visc), 1)[0]', 'dp': 3},
    '9.6': {'h': 'bf3c130df69182e7', 'hint': 'return round(sum(pulls) / len(pulls) / normal, 3)', 'ok': 'Your tool works for any set of pulls.', 'dp': 3, 'args': [([6.0, 6.2, 6.4], 20), ([2.3, 2.4, 2.5], 20), ([10, 11, 12], 40)]},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
### 🔬 Online lab for this unit (free open educational resources)
- **[PhET Friction](https://phet.colorado.edu/en/simulations/friction)** – rub two surfaces and watch the atoms heat up and break away (wear).
- **[Viscosity by Stokes' method (Virtual Labs, NITK Surathkal)](https://ph1-nitk.vlabs.ac.in/exp/viscosity-stokes/)** – compare how fast balls fall through liquids of different viscosity.

Open the online lab in a new tab, try it for a few minutes, then come back here. **Label every reading you take there as "simulated".**

**Key ideas.** A lubricant separates surfaces, so friction and wear fall. A lubricant must be viscous enough to keep a film but not so viscous that it wastes energy. Most oils become **less** viscous as they warm, often close to η = A e^(−bT). Power lost to friction is P = Fv.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
dry = [6.0, 6.2, 6.4]       # N, pulling force on a block, normal force 20 N (illustrative)
oiled = [2.3, 2.4, 2.5]
print('mean dry pull =', np.mean(dry), 'N   mean oiled pull =', np.mean(oiled), 'N')

## Part B – Practise (about 20 minutes)
**B1.** With a normal force of 20 N, find μ for the dry and oiled surfaces above. Give `(mu_dry, mu_oiled)` to 2 decimal places.

In [ ]:
answer_9_1 = ...

In [ ]:
check('9.1', answer_9_1)

In [ ]:
answer_9_2 = ...   # percentage reduction in friction from dry to oiled (1 decimal place)

In [ ]:
check('9.2', answer_9_2)

**B3. Viscosity and temperature.** An engine oil (illustrative data) has these viscosities. By what percentage does the viscosity fall from 20 °C to 80 °C (1 decimal place)?

In [ ]:
temp = np.array([20, 40, 60, 80])                    # °C   (illustrative)
visc = np.array([0.250, 0.150, 0.090, 0.060])        # Pa s
plt.plot(temp, visc, 'o-'); plt.xlabel('temperature (°C)'); plt.ylabel('viscosity (Pa s)'); plt.show()

In [ ]:
answer_9_3 = ...

In [ ]:
check('9.3', answer_9_3)

In [ ]:
answer_9_4 = ...   # power (W) lost when a 15 N friction force acts at a bearing surface moving at 2.0 m/s

In [ ]:
check('9.4', answer_9_4)

**B5. Exponential fit.** Fit ln(η) against T for the oil data. The gradient is −b. Give b (per °C) to 3 decimal places.

In [ ]:
answer_9_5 = ...

In [ ]:
check('9.5', answer_9_5)

## Part C – Build a tool (about 20 minutes)
Complete `friction_coefficient(pulls, normal)`. It should return the mean pull divided by the normal force, rounded to 3 decimal places. Then the cell compares surfaces.

In [ ]:
def friction_coefficient(pulls, normal):
    ...      # YOUR CODE

In [ ]:
check('9.6', friction_coefficient)

In [ ]:
for name, pulls in [('dry', dry), ('oiled', oiled)]:
    print(name, friction_coefficient(pulls, 20))

## Part D – Your own data and reflection (about 10 minutes)
**D1. Prepares you for the Unit 9 assignment.** Repeat the Lab 8 book-on-a-board test with the board dry and then lightly dusted with talc or flour (no oils at home), or use the online labs. Compare μ with your tool and label the data clearly.

In [ ]:
my_dry = [...]
my_lubricated = [...]

**D2. Reflect.**
1. Why can too thick a lubricant waste energy?
2. What is the difference between hydrodynamic and hydrostatic lubrication?
3. Why do bearings get hot when lubrication fails?

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `phy107-labs` repository → commit message *"Lab 9 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.